# 04 - Interior and Exterior Representations

**Goal:** Understand the two standard ways to describe a polyhedron and convert between them in small two-dimensional examples.

A polyhedron can be described either by the halfspaces that contain it or by the points and directions that generate it. In this notebook we will:

1. Introduce **H-representations** and **V-representations**
2. Connect both descriptions through the Minkowski–Weyl theorem
3. Convert a bounded 2D polygon in both directions
4. See why unbounded polyhedra need **extreme rays** in addition to vertices

## Two views of the same polyhedron

### Exterior description: the H-representation

An **H-representation** describes a polyhedron as the intersection of finitely many closed halfspaces:

$$
P = \{x \in \mathbb{R}^n \mid Ax \le b\}.
$$

Every row $a_i^\top$ of $A$ defines one inequality $a_i^\top x \le b_i$. Geometrically, the equation $a_i^\top x = b_i$ defines a boundary hyperplane, and the inequality selects one of its two sides.

This is sometimes called an *exterior representation*: we describe the set by boundaries imposed from the outside.

### Interior description: the V-representation

A **V-representation** describes the same set using finitely many points $v^1,\ldots,v^k$ and directions $r^1,\ldots,r^\ell$:

$$
P = \operatorname{conv}\{v^1,\ldots,v^k\}
    + \operatorname{cone}\{r^1,\ldots,r^\ell\}.
$$

Equivalently, every $x \in P$ can be written as

$$
\begin{aligned}
x &= \sum_{i=1}^k \lambda_i v^i + \sum_{j=1}^{\ell} \mu_j r^j, \\
\sum_{i=1}^k \lambda_i &= 1, \\
\lambda_i &\ge 0, \qquad \mu_j \ge 0.
\end{aligned}
$$

The coefficients $\lambda$ form a **convex combination** of points. The coefficients $\mu$ form a **conic combination** of directions and may grow without bound.

## The Minkowski–Weyl theorem

The bridge between the two descriptions is the **Minkowski–Weyl theorem**:

> A set $P \subseteq \mathbb{R}^n$ is a polyhedron if and only if it can be written as the sum of the convex hull of finitely many points and the conic hull of finitely many directions.

In short,

$$
\boxed{
P = \{x \mid Ax \le b\}
\quad \Longleftrightarrow \quad
P = \operatorname{conv}(V) + \operatorname{cone}(R)
}
$$

For a **bounded** polyhedron (a *polytope*), there are no unbounded directions, so the representation simplifies to

$$
P = \operatorname{conv}(V).
$$

In a minimal description, $V$ contains the vertices and $R$ contains the extreme rays. Descriptions do not have to be minimal: redundant inequalities, interior points, and redundant rays may all be present.

## From inequalities to vertices in 2D

Consider the bounded polygon

$$
P = \left\{(x_1,x_2) \in \mathbb{R}^2 \ \middle|\
\begin{aligned}
x_1 &\ge 0, & x_2 &\ge 0, \\
x_1 &\le 4, & x_2 &\le 3, \\
x_1+x_2 &\le 5
\end{aligned}
\right\}.
$$

Its H-representation is $P=\{x\mid Ax\le b\}$ with the following matrix and right-hand side:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src import hrep_to_vertices_2d, vertices_to_hrep_2d, visualize_polyhedron

A = np.array(
    [
        [-1.0, 0.0],  # x1 >= 0
        [0.0, -1.0],  # x2 >= 0
        [1.0, 0.0],   # x1 <= 4
        [0.0, 1.0],   # x2 <= 3
        [1.0, 1.0],   # x1 + x2 <= 5
    ]
)
b = np.array([0.0, 0.0, 4.0, 3.0, 5.0]).reshape(-1, 1)

visualize_polyhedron(
    A,
    b,
    xlim=(-1, 5),
    ylim=(-1, 4),
    title="A polygon given by its H-representation",
)

In two dimensions, a vertex usually occurs where two non-parallel boundary lines meet. This suggests a direct algorithm:

1. Select every pair of inequalities.
2. Replace both inequalities by equalities and solve the resulting $2\times2$ system.
3. Keep the intersection if it satisfies **all** inequalities.
4. Remove duplicates and order the remaining points around the polygon.

This is a transparent method for small examples. Its pair enumeration is already quadratic in the number of inequalities, so higher-dimensional software uses more sophisticated vertex-enumeration algorithms.

### Implementation

```{literalinclude} ../src/polyhedra/representations.py
:language: python
:linenos:
:start-after: "# --- REPRESENTATIONS:h_to_v_2d:start"
:end-before: "# --- REPRESENTATIONS:h_to_v_2d:end"
```

In [ ]:
vertices = hrep_to_vertices_2d(A, b)

print("Vertices in counterclockwise order:")
print(vertices)

The resulting V-representation is

$$
P = \operatorname{conv}\left\{
\begin{pmatrix}0\\0\end{pmatrix},
\begin{pmatrix}4\\0\end{pmatrix},
\begin{pmatrix}4\\1\end{pmatrix},
\begin{pmatrix}2\\3\end{pmatrix},
\begin{pmatrix}0\\3\end{pmatrix}
\right\}.
$$

Notice that not every pair of boundary lines creates a vertex. For example, $x_1=4$ and $x_2=3$ intersect at $(4,3)$, but this point violates $x_1+x_2\le5$ and is therefore discarded.

## From vertices to inequalities in 2D

Now suppose only the vertices are known. The convex hull is the smallest convex set containing them. Each edge of a 2D convex hull lies on a **supporting line**, and the polygon lies completely in one of the two associated halfspaces.

A convex-hull algorithm therefore gives us both:

- the vertices on the boundary, and
- one valid inequality for each hull edge.

The implementation below uses SciPy's convex hull routine. Points in the interior and duplicate points are harmless: they do not become hull vertices.

### Implementation

```{literalinclude} ../src/polyhedra/representations.py
:language: python
:linenos:
:start-after: "# --- REPRESENTATIONS:v_to_h_2d:start"
:end-before: "# --- REPRESENTATIONS:v_to_h_2d:end"
```

In [ ]:
A_from_vertices, b_from_vertices = vertices_to_hrep_2d(vertices)

print("Recovered inequalities [A | b]:")
print(np.hstack([A_from_vertices, b_from_vertices]))

max_violation = np.max(A_from_vertices @ vertices.T - b_from_vertices)
print(f"Largest inequality violation among the vertices: {max_violation:.2e}")

The recovered inequalities may look different from the original ones:

- rows can appear in another order;
- an inequality can be multiplied by any positive scalar;
- redundant inequalities from the original description need not reappear.

These are syntactic differences only. What matters is whether both systems describe the same set.

In [ ]:
visualize_polyhedron(
    A_from_vertices,
    b_from_vertices,
    xlim=(-1, 5),
    ylim=(-1, 4),
    title="The H-representation recovered from the vertices",
)

## Why unbounded polyhedra need rays

Vertices alone cannot describe an unbounded set. Consider the non-negative orthant in two dimensions:

$$
P = \{x \in \mathbb{R}^2 \mid x_1\ge0,\ x_2\ge0\}.
$$

Its only vertex is the origin, but the origin alone clearly does not generate the whole set. We also need the directions

$$
r^1=\begin{pmatrix}1\\0\end{pmatrix},
\qquad
r^2=\begin{pmatrix}0\\1\end{pmatrix}.
$$

Thus,

$$
P = \operatorname{conv}\left\{\begin{pmatrix}0\\0\end{pmatrix}\right\}
  + \operatorname{cone}\left\{
      \begin{pmatrix}1\\0\end{pmatrix},
      \begin{pmatrix}0\\1\end{pmatrix}
    \right\}.
$$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
view_limit = 4.0
ax.fill_between([0, view_limit], 0, view_limit, alpha=0.15, label="P (clipped to the view)")
ax.scatter([0], [0], color="black", zorder=3, label="vertex")
ax.arrow(0, 0, 2.8, 0, width=0.035, length_includes_head=True, color="tab:red")
ax.arrow(0, 0, 0, 2.8, width=0.035, length_includes_head=True, color="tab:red", label="extreme rays")
ax.set_xlim(-0.5, view_limit)
ax.set_ylim(-0.5, view_limit)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("An unbounded V-representation")
ax.grid(True)
ax.legend()
plt.show()

The directions that keep us inside an H-represented polyhedron form its **recession cone**:

$$
\operatorname{rec}(P)
= \{r \in \mathbb{R}^n \mid x+\mu r \in P\ \forall x\in P,\ \mu\ge0\}
= \{r \in \mathbb{R}^n \mid Ar\le0\}.
$$

This identity explains why the right-hand side disappears for directions: if $Ax\le b$ and $Ar\le0$, then

$$
A(x+\mu r)=Ax+\mu Ar\le b
\qquad \forall\mu\ge0.
$$

The two helper functions in this notebook intentionally reject unbounded input or ray-based input. Their scope is the bounded 2D case, where the geometry is easiest to inspect.

## How general conversion works

The bounded 2D algorithms are simple, but the ideas from the previous notebooks explain the general case.

### V-representation $\rightarrow$ H-representation

Start from

$$
x = V^\top\lambda + R^\top\mu,
\qquad
\mathbf{1}^\top\lambda=1,
\qquad
\lambda,\mu\ge0.
$$

This is a linear system in $(x,\lambda,\mu)$. By writing equalities as pairs of inequalities and projecting out $\lambda$ and $\mu$, [Fourier–Motzkin elimination](02-Fourier-Motzkin-Elimination.ipynb) produces inequalities involving only $x$.

### H-representation $\rightarrow$ V-representation

A common theoretical device is **homogenization**. From $P=\{x\mid Ax\le b\}$ construct the cone

$$
C = \left\{(x,t) \in \mathbb{R}^{n+1}
\ \middle|\ Ax-bt\le0,\ t\ge0\right\}.
$$

Generators of $C$ with $t>0$ can be scaled to $t=1$ and become points of $P$. Generators with $t=0$ become recession directions. This connects the problem to the cone and polarity tools from [Notebook 03](03-Affine-Mappings-and-Polar-Cones.ipynb).

In practical polyhedral computation, algorithms such as the **double-description method** move between inequalities and generators while controlling redundancies. The number of inequalities or generators can be exponential, so neither representation is always compact.

## Takeaways

- An **H-representation** describes a polyhedron as an intersection of halfspaces: $P=\{x\mid Ax\le b\}$.
- A **V-representation** describes it using points and directions: $P=\operatorname{conv}(V)+\operatorname{cone}(R)$.
- The Minkowski–Weyl theorem says these are two equivalent views of polyhedra.
- For bounded polyhedra, vertices are sufficient: $P=\operatorname{conv}(V)$.
- Unbounded polyhedra additionally need rays, which generate the recession cone $\{r\mid Ar\le0\}$.
- In 2D, we can learn both conversions by intersecting boundary lines and computing supporting lines of a convex hull.
- General conversion is a genuine algorithmic problem and can create exponentially many inequalities or generators.

## Next

The V-representation emphasizes vertices, while integer optimization emphasizes integer points. The next notebook combines these ideas through the **integer hull**:

$$
P_I = \operatorname{conv}(P\cap\mathbb{Z}^n).
$$

➡️ [05 - The Integer Hull](05-The-Integer-Hull.ipynb)